# Converter vs Non-converter MEG Connectivity Analysis

This notebook is the reporting layer for the current TFG dataset.
It now matches the uploaded data layout in `data/` and the real subject naming convention used inside the `.mat` files.

The workflow now does the following:
- discovers subjects by iterating over top-level structs inside each `C_p*.mat` and `NC_p*.mat`
- treats `C_*` as `Converter` and `NC_*` as `Non-converter`
- reconstructs subject-level trial matrices from `Value` using the atlas scout count
- computes PLV, AEC, and orthogonalized AEC for every subject and band
- runs group-level inference with Welch t-tests, BH-FDR, and max-stat permutation correction


## Data Model Confirmed Against the Uploaded Files

The loader is now aligned with the actual dataset structure:

- each `.mat` file contains many subject structs rather than one subject per file
- the subject structs use fields `Value`, `Time`, and `Atlas`
- `Value` has shape `(102 * n_trials, 8000)`
- the 102 ROI labels come from `Atlas.Scouts[*].Label`
- the central analysis window is `2000:6000`, which keeps the clean 4-second segment and discards the 2-second padding before and after

This matches the dataset description you provided: the source-space signals are already reconstructed, collapsed to 102 Schaefer parcels, filtered in the broad range `[0.5, 45] Hz`, and segmented into artifact-free 8-second trials with a 4-second core interval of interest.


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from meg_alzheimer.dataset import discover_subjects
from meg_alzheimer.pipeline import PipelineConfig, run_group_pipeline


In [ ]:
DATA_ROOT = Path('data')
OUTPUT_ROOT = Path('outputs')

config = PipelineConfig(
    data_root=DATA_ROOT,
    output_root=OUTPUT_ROOT,
    sampling_rate=1000.0,
    crop_start=2000,
    crop_end=6000,
    fdr_q=0.10,
    n_perm=1000,
    quicklook_band='alpha',
    quicklook_metric='PLV',
    group_a='Converter',
    group_b='Non-converter',
    save_subject_graphs=True,
)

config


In [ ]:
subjects = discover_subjects(DATA_ROOT)
subject_table = pd.DataFrame(
    [
        {
            'subject_id': s.subject_id,
            'group': s.group,
            'source_file': s.path.name,
            'mat_variable': s.mat_variable,
        }
        for s in subjects
    ],
    columns=['subject_id', 'group', 'source_file', 'mat_variable'],
)

print(f'Discovered {len(subject_table)} subjects under {DATA_ROOT.resolve()}')
if not subject_table.empty:
    display(subject_table.groupby('group').size().rename('n_subjects').reset_index())
    display(subject_table.groupby('source_file').size().rename('n_subjects').reset_index())
    display(subject_table.head(15))

if not (DATA_ROOT / 'NC_p3.mat').exists():
    print('Note: NC_p3.mat is not present yet, so the Non-converter cohort is currently incomplete.')


## Metric Rationale

Three complementary connectivity indices are retained:

- `PLV`: phase synchronization between ROIs
- `AEC`: envelope correlation, useful but leakage-sensitive
- `AEC-orth`: orthogonalized envelope correlation, included as the more conservative amplitude-based metric

Keeping all three is important here because the signals already come from source space, but source leakage can still bias amplitude-coupling estimates.
AEC-orth is therefore not a cosmetic addition; it is the metric that helps distinguish plausible large-scale coupling from zero-lag mixing artifacts.


## Run the Batch Pipeline

The pipeline processes one `.mat` file at a time and extracts all subject structs from that file.
This is necessary for the current repository because the uploaded files are large and each file packs many subjects.

For every discovered subject, the pipeline:
- reshapes `Value` into `trials x rois x time`
- crops the central clean interval `2000:6000`
- computes band-limited connectivity trial by trial and averages across trials
- saves subject-level matrices and quicklook figures
- aggregates results into group-level tables and inferential outputs


In [ ]:
if subject_table.empty:
    results = {
        'subjects': [],
        'message': f'No dataset was found under {DATA_ROOT.resolve()}.',
        'output_root': str(OUTPUT_ROOT.resolve()),
    }
    print(results['message'])
else:
    results = run_group_pipeline(config)

results


## Output Tables

The pipeline exports several result layers:

- `subjects.csv`: one row per subject struct discovered inside the `.mat` files
- `subject_global_means.csv`: global upper-triangle means for every subject, band, and metric
- `subject_network_means.csv`: intra-network and inter-network summaries
- `global_group_stats.csv`: group-level tests on global means
- `network_group_stats.csv`: group-level tests on network summaries with FDR over network comparisons
- `group_stats/*.npz`: full edgewise maps, correction masks, and effect sizes


In [ ]:
table_paths = {
    'subjects': OUTPUT_ROOT / 'subjects.csv',
    'subject_global_means': OUTPUT_ROOT / 'subject_global_means.csv',
    'subject_network_means': OUTPUT_ROOT / 'subject_network_means.csv',
    'global_group_stats': OUTPUT_ROOT / 'global_group_stats.csv',
    'network_group_stats': OUTPUT_ROOT / 'network_group_stats.csv',
    'edgewise_group_summary': OUTPUT_ROOT / 'edgewise_group_summary.csv',
}

loaded_tables = {}
for name, path in table_paths.items():
    if path.exists():
        loaded_tables[name] = pd.read_csv(path)
        print(f'{name}: {path} ({len(loaded_tables[name])} rows)')
    else:
        print(f'{name}: missing ({path})')


In [ ]:
if 'global_group_stats' in loaded_tables:
    display(loaded_tables['global_group_stats'].sort_values('p_value'))

if 'network_group_stats' in loaded_tables:
    display(loaded_tables['network_group_stats'].sort_values('p_value').head(15))

if 'edgewise_group_summary' in loaded_tables:
    display(loaded_tables['edgewise_group_summary'].sort_values(['n_fdr_edges', 'n_perm_edges'], ascending=False))


## Inspect Edgewise Group Maps

Each edgewise `.npz` bundle contains:
- `t_stat`
- `p_value`
- `fdr_mask`
- `perm_p_value`
- `perm_significant_mask`
- `effect_size_d`

The contrast is always reported as `group_a - group_b`, which in the default configuration means `Converter - Non-converter`.


In [ ]:
stats_file = OUTPUT_ROOT / 'group_stats' / f"{config.quicklook_band}_{config.quicklook_metric.lower().replace('-', '_')}_edgewise_stats.npz"

if stats_file.exists():
    stats_bundle = np.load(stats_file)
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))

    im0 = axes[0].imshow(stats_bundle['t_stat'], origin='lower', cmap='coolwarm')
    axes[0].set_title(f'{config.group_a} - {config.group_b} t-statistics')
    plt.colorbar(im0, ax=axes[0], fraction=0.046, pad=0.04)

    im1 = axes[1].imshow(stats_bundle['fdr_mask'].astype(float), origin='lower', vmin=0, vmax=1, cmap='magma')
    axes[1].set_title('FDR-significant edges')
    plt.colorbar(im1, ax=axes[1], fraction=0.046, pad=0.04)

    im2 = axes[2].imshow(stats_bundle['perm_significant_mask'].astype(float), origin='lower', vmin=0, vmax=1, cmap='magma')
    axes[2].set_title('Permutation-significant edges')
    plt.colorbar(im2, ax=axes[2], fraction=0.046, pad=0.04)

    for ax in axes:
        ax.set_xlabel('ROI')
        ax.set_ylabel('ROI')
    plt.tight_layout()
    plt.show()
else:
    print(f'No edgewise stats file found at {stats_file}')


In [ ]:
subject_npz_paths = sorted((OUTPUT_ROOT / 'subjects').glob('*/*/connectivity_matrices.npz'))
key = f"{config.quicklook_band}__{config.quicklook_metric}"

if subject_npz_paths:
    first_subject_path = subject_npz_paths[0]
    subject_bundle = np.load(first_subject_path)
    if key in subject_bundle:
        plt.figure(figsize=(6, 5))
        plt.imshow(subject_bundle[key], origin='lower', cmap='viridis')
        plt.colorbar(fraction=0.046, pad=0.04)
        plt.title(f'Quicklook subject matrix: {first_subject_path.parent.name} ({key})')
        plt.xlabel('ROI')
        plt.ylabel('ROI')
        plt.tight_layout()
        plt.show()
    else:
        print(f'{key} not found inside {first_subject_path}')
else:
    print('No subject-level connectivity bundles were found in outputs/subjects')


## Methodological Limits

This notebook is now aligned with the real file naming and storage layout, but several scientific limits remain:

- the data are assumed to be already clean enough for connectivity analysis
- the broad-band filtering and source reconstruction happened before this repository's pipeline starts
- the current implementation intentionally does not add a new QC layer because that was explicitly excluded from the requested changes
- until `NC_p3.mat` is uploaded, the Non-converter cohort is incomplete in this checkout

The engineering side is now consistent with the uploaded dataset. The remaining uncertainty is therefore not about file parsing anymore, but about cohort completeness and preprocessing quality.
